# What we want this dataset to look like :

---

In [ ]:
import numpy as np
import xarray as xr
import zarr

# -----------------------------
# Parameters
# -----------------------------
zarr_path = "./latent_forecast.zarr"   # change to s3://... later if needed

rollout_steps = 10
n_spatial = 259_200
n_feature = 1024

init_times = np.arange(
    np.datetime64("2025-07-01T00"),
    np.datetime64("2025-07-04T00"),
    np.timedelta64(6, "h"),
)

# -----------------------------
# Coordinates
# -----------------------------
lead_time = (np.arange(1, rollout_steps + 1) * 6).astype("int64")

valid_time = (
    init_times[:, None]
    + lead_time[None, :] * np.timedelta64(1, "h")
)

# -----------------------------
# Coordinate-only Dataset
# -----------------------------
ds = xr.Dataset(
    coords={
        "init_time": ("init_time", init_times),
        "lead_time": ("lead_time", lead_time),
        "spatial_location": (
            "spatial_location",
            np.arange(n_spatial, dtype="int64"),
        ),
        "feature": (
            "feature",
            np.arange(n_feature, dtype="int64"),
        ),
        "valid_time": (
            ("init_time", "lead_time"),
            valid_time,
        ),
    },
    attrs={
        "description": "Aurora latent forecast dataset",
        "schema_version": "v1",
        "rollout_steps": rollout_steps,
        "temporal_semantics": "valid_time = init_time + lead_time",
    },
)

# -----------------------------
# Write metadata only
# -----------------------------
ds.to_zarr(
    zarr_path,
    zarr_format=3,
    mode="w",
    consolidated=False,
    write_empty_chunks=False,   # critical
)

# -----------------------------
# Create empty array (schema only)
# -----------------------------
zarr.create_array(
    zarr_path,
    name="latent_forecast",
    shape=(
        len(init_times),
        len(lead_time),
        n_spatial,
        n_feature,
    ),
    dtype="float32",
    fill_value=np.nan,
    dimension_names=(
        "init_time",
        "lead_time",
        "spatial_location",
        "feature",
    ),
)

print("Zarr schema initialized (metadata-only)")

xr.open_zarr(zarr_path, zarr_format=3, consolidated=False, chunks=None)


# Source Data
---

In [ ]:
import kafou_arraylake as arraylake
import zarr
import numpy as np
import xarray as xr

repo_name = "kafou/aurora-era5-samples"
branch = "extend-2025"

client = arraylake.Client()
repo = client.get_repo(repo_name)
ro = repo.readonly_session(branch)
ds = xr.open_zarr(
    ro.store,
    group="samples",
    zarr_format=3,
    consolidated=False,
    chunks=None,
)

print(ds)



# Check 

---

In [ ]:
import numpy as np
import xarray as xr
import kafou_arraylake as arraylake

repo_name = "kafou/aurora-era5-forecast-latent-vectors-november"
branch = "main"

client = arraylake.Client()
repo = client.get_repo(repo_name)
ro = repo.readonly_session(branch)

ds = xr.open_zarr(
    ro.store,
    zarr_format=3,
    consolidated=False,
    chunks=None,
)

print(ds)

init_time = np.datetime64("2024-11-01T00:00:00")
lead_time = 6  # hours

slab = ds["lv"].sel(
    init_time=init_time,
    lead_time=lead_time,
)

value = slab.isel(spatial_location=100, feature=600).values
print(value)



In [ ]:
import numpy as np
import xarray as xr
import kafou_arraylake as arraylake

repo_name = "kafou/aurora-era5-forecast-latent-vectors-november"
branch = "main"

client = arraylake.Client()
repo = client.get_repo(repo_name)
ro = repo.readonly_session(branch)

ds = xr.open_zarr(
    ro.store,
    zarr_format=3,
    consolidated=False,
    chunks=None,
)

print(ds)

init_time = np.datetime64("2024-11-27T00:00:00")
lead_time = 6  # hours

slab = ds["lv"].sel(
    init_time=init_time,
    lead_time=lead_time,
)

value = slab.isel(spatial_location=100, feature=600).values
print(value)



# Covnert to operational_example format

---

In [12]:
import xarray as xr
import numpy as np

# 1. Select daily 0Z init_times and keep only the first 2
ds_4z = ds.sel(init_time=ds.init_time.dt.hour == 0)
ds_4z = ds_4z.isel(init_time=slice(0, 2))

# 2. Keep only the first 4 rollout lead times
ds_roll4 = ds_4z.sel(lead_time=[6, 12, 18, 24])

# 3. Flatten (init_time, lead_time) → time using valid_time
lv_time = ds_roll4["lv"].stack(time=("init_time", "lead_time"))

# 4. Remove MultiIndex + old coordinates
lv_time = (
    lv_time
    .reset_index("time")                # MultiIndex → variable
    .drop_vars(["init_time", "lead_time"])
    .assign_coords(time=lambda x: x["time"])  # promote to coord
)

# 5. Final inference-style dataset (DO NOT re-add coords)
ds_final = lv_time.to_dataset(name="lv")


In [ ]:
print(ds_final)

In [ ]:
import numpy as np

lat_range = (49.0, 61.0)
lon_range = (-8.5, 2.5)   # crosses 0°

n = (
    (np.linspace(89.5, -89.5, 180) >= lat_range[0])
    & (np.linspace(89.5, -89.5, 180) <= lat_range[1])
).sum() * (
    (np.linspace(-179.5, 179.5, 360) >= lon_range[0])
    & (np.linspace(-179.5, 179.5, 360) <= lon_range[1])
).sum()

4 * n

In [1]:
import numpy as np

def get_spatial_indices_from_bounds(
    *,
    lat_range: tuple[float, float],
    lon_range: tuple[float, float],
    n_lev: int = 4,
    n_lat: int = 180,
    n_lon: int = 360,
) -> np.ndarray:
    """
    Return ORIGINAL spatial_location indices for a lat/lon bounding box.
    Longitude bounds must be Greenwich-centered [-180, 180].
    Level-inclusive.
    """

    # grid centers
    lats = np.linspace(89.5, -89.5, n_lat)
    lons = np.linspace(0.5, 359.5, n_lon)
    lons = ((lons + 180) % 360) - 180  # normalize values ONLY

    flat = np.arange(n_lev * n_lat * n_lon)

    lev = flat // (n_lat * n_lon)
    rem = flat % (n_lat * n_lon)
    lat_idx = rem // n_lon
    lon_idx = rem % n_lon

    lat_vals = lats[lat_idx]
    lon_vals = lons[lon_idx]

    mask = (
        (lat_vals >= lat_range[0])
        & (lat_vals <= lat_range[1])
        & (
            (lon_vals >= lon_range[0]) & (lon_vals <= lon_range[1])
            if lon_range[0] <= lon_range[1]
            else (lon_vals >= lon_range[0]) | (lon_vals <= lon_range[1])
        )
    )

    spatial_indices = flat[mask].astype(np.int64)

    return spatial_indices


In [ ]:
import xarray as xr
import icechunk
import zarr
from datetime import datetime
import datetime

start_time = "2024-01-01T00:00:00"
end_time = "2024-01-17T18:00:00"

start_time = datetime.datetime.fromisoformat(start_time)
end_time = datetime.datetime.fromisoformat(end_time)


# Build init_time axis (pure datetime)
init_times = []
this_time = start_time
while this_time <= end_time:
    init_times.append(this_time)
    this_time += datetime.timedelta(hours=6)


rollout_steps= 7
lat_range =  (49.0, 61.0)
lon_range = (-8.5, 2.5)
n_feature = 1024

# calculate spatial size
if lat_range and lon_range:
    spatial_indices = get_spatial_indices_from_bounds(
        lat_range=lat_range,
        lon_range=lon_range,
    )
    n_spatial = len(spatial_indices)

else:
    n_spatial = 259200

print(n_spatial)

init_times = np.asarray(init_times, dtype="datetime64[ns]")
lead_times = (np.arange(1, rollout_steps + 1) * 6).astype("int64")  # hours
valid_times = init_times[:, None] + lead_times[None, :] * np.timedelta64(1, "h")

coord_ds = xr.Dataset(
    coords={
        "init_time": ("init_time", init_times),
        "lead_time": ("lead_time", lead_times),
        "spatial_location": ("spatial_location", spatial_indices),
        "feature": ("feature", np.arange(n_feature, dtype="int64")),
        "valid_time": (("init_time", "lead_time"), valid_times),
    },
    attrs={
        "description": "Aurora latent forecast dataset",
        "rollout_steps": int(rollout_steps),
        "schema_version": "v1",
    },
)

display(coord_ds)

print("[INIT] init_time dtype:", coord_ds["init_time"].dtype)
print("[INIT] init_time head:", coord_ds["init_time"].values[:3])
print("[INIT] lead_time:", coord_ds["lead_time"].values)

# coord_ds.to_zarr(
#     store,
#     zarr_format=3,
#     mode="w",
#     consolidated=False,
#     write_empty_chunks=False,
#     # keep chunk hints modest; coords are tiny except spatial_location
#     encoding={
#         "init_time": {"chunks": (len(init_times),)},
#         "lead_time": {"chunks": (len(lead_times),)},
#         "valid_time": {"chunks": (len(init_times), len(lead_times))},
#         "spatial_location": {"chunks": (n_spatial,)},
#         "feature": {"chunks": (n_feature,)},
#     },
# )

# # Create empty latent array (metadata + NaN fill)
# zarr.create_array(
#     store,
#     name="lv",
#     shape=(len(init_times), len(lead_times), n_spatial, n_feature),
#     chunks=(1, len(lead_times), 1024, 128),  # tune later
#     dtype="float32",
#     fill_value=np.nan,
#     compressors=[],
#     dimension_names=("init_time", "lead_time", "spatial_location", "feature"),
# )

print("[INIT] wrote coords + empty lv")


In [14]:

def recenter_to_greenwich(ds, *, n_lat=180, n_lon=360):
    """Recenter a flattened (lev, lat, lon) grid from dateline-centered (0–360)
    to Greenwich-centered (-180–180).

    Assumes spatial_location ordering:
        lev-major, then lat-major, then lon-major.
    """
    # normalize longitude to [-180, 180)
    lon_new = ((ds.lon.values + 180) % 360) - 180

    flat = np.arange(ds.sizes["spatial_location"])
    lev = flat // (n_lat * n_lon)
    rem = flat % (n_lat * n_lon)
    lat_i = rem // n_lon
    lon_i = rem % n_lon

    # correct stable ordering: lev → lat → lon_value
    order = np.lexsort((lon_new, lat_i, lev))

    return ds.isel(spatial_location=order).assign_coords(lon=("spatial_location", lon_new[order]))


def subset_lv_dataset(
    ds: xr.Dataset,
    *,
    lat_range: tuple[float, float],
    lon_range: tuple[float, float],
) -> xr.Dataset:
    """Spatial subset ONLY.

    Keeps:
      - ALL times
      - ALL levels
      - ALL features
      - spatial_location dimension intact

    Parameters
    ----------
    ds : xr.Dataset
        Input dataset. Assumes a single dimension 'spatial_location' that is a flattened
        (lev, lat, lon) grid, with lev-major, then lat-major, then lon-major ordering.
        (i.e., for each level, all latitudes, for each latitude, all longitudes).
    lat_range : tuple of float
        (min_lat, max_lat) in degrees, specifying the latitude bounds for subsetting.
        Must be in Greenwich-centered coordinates (i.e., -90 to 90, north positive).
    lon_range : tuple of float
        (min_lon, max_lon) in degrees, specifying the longitude bounds for subsetting.
        Must be in Greenwich-centered coordinates (i.e., -180 to 180, east positive).

    Notes:
    -----
    - The function expects the input dataset to use dateline-centered (0–360) longitudes.
    - It automatically recenters the longitude coordinates to Greenwich-centered (-180–180).
    - The function is memory-safe.
    """
    # GRID SHAPE (MUST MATCH DATA)
    n_lev = 4
    n_lat = 180
    n_lon = 360

    # latitude centers (north → south)
    lats = np.linspace(89.5, -89.5, n_lat)

    # longitude centers (0.5 → 359.5, dateline-centered)
    lons = np.linspace(0.5, 359.5, n_lon)

    # RECONSTRUCT FLATTENED INDEXING
    # spatial_location = lev-major, lat-major, lon-major
    flat = np.arange(ds.sizes["spatial_location"])

    lev = flat // (n_lat * n_lon)
    rem = flat % (n_lat * n_lon)
    lat_idx = rem // n_lon
    lon_idx = rem % n_lon

    lat_vals = lats[lat_idx]
    lon_vals = lons[lon_idx]

    # ATTACH COORDINATES
    ds = ds.assign_coords(
        lat=("spatial_location", lat_vals),
        lon=("spatial_location", lon_vals),
        lev=("spatial_location", lev),
    ).set_coords(["lat", "lon", "lev"])

    ds_greenwich = recenter_to_greenwich(ds)

    print("SUCCESS: lat / lon / lev coordinates attached")
    print("lat range:", float(ds_greenwich.lat.min()), float(ds_greenwich.lat.max()))
    print("lon range:", float(ds_greenwich.lon.min()), float(ds_greenwich.lon.max()))
    print("levels:", np.unique(ds_greenwich.lev.values))

    mask = (
        (ds_greenwich.lat >= lat_range[0])
        & (ds_greenwich.lat <= lat_range[1])
        & (ds_greenwich.lon >= lon_range[0])
        & (ds_greenwich.lon <= lon_range[1])
    )

    ds_sub = ds_greenwich.isel(spatial_location=mask)

    ds_sub.attrs.update(
        dict(
            #valid_time_range=["2009-01-01T12:00:00", ds_sub.time.values.max().astype(str)],
            lat_range=lat_range,
            lon_range=lon_range,
            created_by="Zora Zorkic",
        )
    )

    return ds_sub




In [ ]:
import xarray as xr
import icechunk
import zarr
from datetime import datetime
import datetime

start_time = "2024-01-01T00:00:00"
end_time = "2024-01-17T18:00:00"

start_time = datetime.datetime.fromisoformat(start_time)
end_time = datetime.datetime.fromisoformat(end_time)


# Build init_time axis (pure datetime)
init_times = []
this_time = start_time
while this_time <= end_time:
    init_times.append(this_time)
    this_time += datetime.timedelta(hours=6)


rollout_steps= 7
# lat_range =  (49.0, 61.0)
# lon_range = (-8.5, 2.5)

lat_range=(28, 32)
lon_range=(-102, -98)

n_feature = 1024

# calculate spatial size
if lat_range and lon_range:
    spatial_indices = get_spatial_indices_from_bounds(
        lat_range=lat_range,
        lon_range=lon_range,
    )
    n_spatial = len(spatial_indices)

else:
    n_spatial = 259200

print(n_spatial)

init_times = np.asarray(init_times, dtype="datetime64[ns]")
lead_times = (np.arange(1, rollout_steps + 1) * 6).astype("int64")  # hours
valid_times = init_times[:, None] + lead_times[None, :] * np.timedelta64(1, "h")

coord_ds = xr.Dataset(
    coords={
        "init_time": ("init_time", init_times),
        "lead_time": ("lead_time", lead_times),
        "spatial_location": ("spatial_location", np.arange(259200, dtype="int64")),
        "feature": ("feature", np.arange(n_feature, dtype="int64")),
        "valid_time": (("init_time", "lead_time"), valid_times),
    },
    attrs={
        "description": "Aurora latent forecast dataset",
        "rollout_steps": int(rollout_steps),
        "schema_version": "v1",
    },
)

coord_ds = subset_lv_dataset(
    ds=coord_ds, lat_range=lat_range,lon_range=lon_range)


display(coord_ds)

print("[INIT] init_time dtype:", coord_ds["init_time"].dtype)
print("[INIT] init_time head:", coord_ds["init_time"].values[:3])
print("[INIT] lead_time:", coord_ds["lead_time"].values)



# coord_ds.to_zarr(
#     store,
#     zarr_format=3,
#     mode="w",
#     consolidated=False,
#     write_empty_chunks=False,
#     # keep chunk hints modest; coords are tiny except spatial_location
#     encoding={
#         "init_time": {"chunks": (len(init_times),)},
#         "lead_time": {"chunks": (len(lead_times),)},
#         "valid_time": {"chunks": (len(init_times), len(lead_times))},
#         "spatial_location": {"chunks": (n_spatial,)},
#         "feature": {"chunks": (n_feature,)},
#     },
# )

# # Create empty latent array (metadata + NaN fill)
# zarr.create_array(
#     store,
#     name="lv",
#     shape=(len(init_times), len(lead_times), n_spatial, n_feature),
#     chunks=(1, len(lead_times), 1024, 128),  # tune later
#     dtype="float32",
#     fill_value=np.nan,
#     compressors=[],
#     dimension_names=("init_time", "lead_time", "spatial_location", "feature"),
# )

print("[INIT] wrote coords + empty lv")


In [1]:
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import kafou_arraylake as arraylake
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import xarray as xr
import zarr
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER




# ============================================================
# 1) ATTACH GEOMETRY (RUN ONCE, AUTHORITATIVE)
# ============================================================

def attach_spatial_geometry(
    ds: xr.Dataset,
    *,
    n_lev: int,
    n_lat: int,
    n_lon: int,
) -> xr.Dataset:
    """
    Attach (lev, lat, lon) coordinates to a flattened spatial_location axis.

    Assumes spatial_location ordering:
        lev-major → lat-major → lon-major

    Latitude convention:
        lat = -90 at top, +90 at bottom (southward-positive)

    Longitude convention:
        lon in [0, 360), prime-meridian centered
    """
    n_expected = n_lev * n_lat * n_lon
    if ds.sizes["spatial_location"] != n_expected:
        raise ValueError(
            f"Expected spatial_location={n_expected}, "
            f"got {ds.sizes['spatial_location']}"
        )

    # ERA5-style cell centers
    lat_vals = np.linspace(-89.5, 89.5, n_lat)   # southward-positive
    lon_vals = np.linspace(0.5, 359.5, n_lon)

    flat = np.arange(n_expected)

    lev = flat // (n_lat * n_lon)
    rem = flat % (n_lat * n_lon)
    lat_i = rem // n_lon
    lon_i = rem % n_lon

    ds = ds.assign_coords(
        lev=("spatial_location", lev.astype("int8")),
        lat=("spatial_location", lat_vals[lat_i]),
        lon=("spatial_location", lon_vals[lon_i]),
    ).set_coords(["lev", "lat", "lon"])

    ds.attrs.update(
        grid_type="flattened_lev_lat_lon",
        grid_shape=(n_lev, n_lat, n_lon),
        lat_convention="southward_positive",
        lon_convention="0_to_360",
    )

    return ds


# ============================================================
# 2) RECENTER LONGITUDE TO GREENWICH (−180..180)
# ============================================================

def recenter_longitude_greenwich(ds: xr.Dataset) -> xr.Dataset:
    """
    Convert longitude from [0, 360) to [-180, 180) and enforce
    stable ordering: lev → lat → lon.
    """
    if "lon" not in ds.coords:
        raise ValueError("Dataset must have lon coordinate")

    lon_new = ((ds.lon + 180) % 360) - 180
    ds = ds.assign_coords(lon=lon_new)

    # stable ordering for plotting + unstacking
    order = np.lexsort(
        (
            ds.lon.values,
            ds.lat.values,
            ds.lev.values,
        )
    )

    ds = ds.isel(spatial_location=order)
    ds.attrs["lon_convention"] = "greenwich"

    return ds


# ============================================================
# 3) SUBSET BY GREENWICH-CENTERED LAT / LON
# ============================================================

def subset_by_latlon(
    ds: xr.Dataset,
    *,
    lat_range: tuple[float, float],
    lon_range: tuple[float, float],
) -> xr.Dataset:
    """
    Spatial subset using semantic coordinates.

    Keeps:
      - all times
      - all levels
      - all features
      - spatial_location values intact (global IDs)
    """
    required = {"lat", "lon", "lev"}
    if not required <= set(ds.coords):
        raise ValueError("Dataset missing lat/lon/lev coordinates")

    mask = (
        (ds.lat >= lat_range[0]) &
        (ds.lat <= lat_range[1]) &
        (ds.lon >= lon_range[0]) &
        (ds.lon <= lon_range[1])
    )

    ds_sub = ds.isel(spatial_location=mask)

    ds_sub.attrs.update(
        lat_range=lat_range,
        lon_range=lon_range,
    )

    return ds_sub


# ============================================================
# 4) SAFE REUSE OF spatial_location ON ANOTHER DATASET
# ============================================================

def subset_other_dataset_by_spatial_location(
    ds_full: xr.Dataset,
    ds_subset: xr.Dataset,
) -> xr.Dataset:
    """
    Subset another lv(time, spatial_location, feature) dataset
    using spatial_location from an existing subset.

    Requires spatial_location to be a shared global index.
    """
    return ds_full.sel(
        spatial_location=ds_subset.spatial_location
    )


# ============================================================
# 5) OPTIONAL: UNFLATTEN FOR PLOTTING / ANALYSIS
# ============================================================

def to_lev_lat_lon_grid(ds: xr.Dataset) -> xr.Dataset:
    """
    Recover (lev, lat, lon) grid from flattened layout.
    Safe for subsets.
    """
    return (
        ds
        .set_index(spatial_location=("lev", "lat", "lon"))
        .unstack("spatial_location")
        .sortby("lat")
        .sortby("lon")
    )


# ============================================================
# 6) SANITY CHECK (RUN ONCE)
# ============================================================

def assert_spatial_compatibility(ds_a: xr.Dataset, ds_b: xr.Dataset):
    """
    Assert that two datasets share the same spatial_location meaning.
    """
    xr.testing.assert_equal(
        ds_a.lat,
        ds_b.lat.sel(spatial_location=ds_a.spatial_location)
    )


def plot_lv_dataset(
    ds: xr.Dataset,
    *,
    time: str,
    lev: int,
    feature: int,
    lat_range: tuple[float, float] | None = None,
    lon_range: tuple[float, float] | None = None,
    cmap: str = "viridis",
    figsize: tuple = (8, 6),
):
    """Plot a single (time, lev, feature) slice from a spatially-subsetted,
    Greenwich-centered latent-vector dataset.

    If lat_range or lon_range is None, it is inferred from the data.
    """
    # --------------------------------------------------
    # 1) SELECT ONE SLICE (cheap)
    # --------------------------------------------------
    da = ds.lv.sel(time=time, feature=feature).where(ds.lev == lev, drop=True)

    lat = ds.lat.where(ds.lev == lev, drop=True).values
    lon = ds.lon.where(ds.lev == lev, drop=True).values

    # --------------------------------------------------
    # 2) UNSTACK TO (lat, lon) GRID (safe)
    # --------------------------------------------------
    da2 = (
        da.assign_coords(
            lat=("spatial_location", lat),
            lon=("spatial_location", lon),
        )
        .set_index(spatial_location=("lat", "lon"))
        .unstack("spatial_location")
        .sortby("lat")
        .sortby("lon")
    )

    grid = da2.values
    lat_u = da2.lat.values
    lon_u = da2.lon.values

    # --------------------------------------------------
    # 3) INFER BOUNDS IF NEEDED
    # --------------------------------------------------
    if lat_range is None:
        lat_range = (float(lat_u.min()) - 0.5, float(lat_u.max()) + 0.5)

    if lon_range is None:
        lon_range = (float(lon_u.min()) - 0.5, float(lon_u.max()) + 0.5)

    # --------------------------------------------------
    # 4) PLOT
    # --------------------------------------------------
    ax = plt.axes(projection=ccrs.PlateCarree())

    ax.set_extent(
        [lon_range[0], lon_range[1], lat_range[0], lat_range[1]],
        crs=ccrs.PlateCarree(),
    )

    ax.add_feature(cfeature.COASTLINE, linewidth=1.0)
    ax.add_feature(cfeature.BORDERS, linewidth=0.8)

    pcm = ax.pcolormesh(
        lon_u,
        lat_u,
        grid,
        shading="nearest",
        cmap=cmap,
        transform=ccrs.PlateCarree(),
    )

    # --------------------------------------------------
    # 5) GRIDLINES + TICKS (1°)
    # --------------------------------------------------
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.5,
        color="gray",
        alpha=0.6,
        linestyle=":",
    )

    gl.top_labels = False
    gl.right_labels = False

    gl.xlocator = mticker.MultipleLocator(1)
    gl.ylocator = mticker.MultipleLocator(1)

    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER

    gl.xlabel_style = {
        "size": 9,
        "rotation": 45,
        "ha": "right",
    }

    gl.ylabel_style = {
        "size": 9,
        "va": "center",
    }

    # --------------------------------------------------
    # 6) FINAL TOUCHES
    # --------------------------------------------------
    plt.colorbar(pcm, ax=ax, label="lv value")

    ax.set_title(
        f"Greenwich-centered LV data\n"
        f"Feature={feature} | Level={lev} | Time={time}\n"
        f"Lat={lat_range} | Lon={lon_range}",
        fontsize=10,
    )

    plt.show()




In [ ]:
import kafou_arraylake as arraylake 
import xarray as xr

SOURCE_REPO = "kafou/aurora-era5-forecast-lv-6z-rollout-geo-uk"
SOURCE_BRANCH = "main"

client = arraylake.Client()
repo = client.get_repo(SOURCE_REPO)
session = repo.readonly_session(SOURCE_BRANCH)

ds_lv = xr.open_zarr(session.store, zarr_format=3, consolidated=False, chunks=None)

print(ds_lv)

lv = ds_lv["lv"].values


init_time = 365 * 10 # days * years
lead_time = 7   # steps
spatial_location = 1024
feature = 1024

lv = init_time * lead_time * spatial_location * feature * 4

((lv / 1024**2 ) / 1024 ) * 1.07374 # GB

In [ ]:
written_mask = (
    ~xr.ufuncs.isnan(ds_lv["lv"])
    .any(dim=("lead_time", "spatial_location", "feature"))
)

written_init_times = ds_lv["init_time"].where(written_mask, drop=True)

written_init_times


In [ ]:
import zarr

lv = zarr.open_array(session.store, path="lv", zarr_format=3, mode="r")

lv[:1, :1, :1, :1]


In [15]:
arry = ds_lv["lv"].isel(init_time=0, lead_time=0).values.reshape(4, 16, 16, 1024)

In [ ]:
import matplotlib.pyplot as plt

field = arry[0, :, :, 622]  # (lat, lon)

plt.figure(figsize=(5, 4))
plt.imshow(field, origin="upper", aspect="auto")
plt.colorbar(label="LV feature 622")
plt.title("Level 0 · Feature 622")
plt.xlabel("Longitude index")
plt.ylabel("Latitude index")
plt.show()
